In [ ]:
import base64
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys

WORK = Path("/kaggle/working").resolve(strict=True)
CODE = WORK / "bh9.py"
ENVELOPE = json.loads("{\"bh9.py\":{\"base64\":\"...\",\"bytes\":230086,\"sha256\":\"28b0916d302e2e219ef7ab790a410860c8e413ab87ea9654f18704331a39ad01\"}}")
if set(ENVELOPE) != {"bh9.py"} or CODE.exists() or CODE.parent.resolve(strict=True) != WORK:
    raise RuntimeError("one-trial standalone code path is invalid")
row = ENVELOPE["bh9.py"]
raw = base64.b64decode(row["base64"], validate=True)
if len(raw) != row["bytes"] or hashlib.sha256(raw).hexdigest() != row["sha256"]:
    raise RuntimeError("embedded code_file hash mismatch")
with CODE.open("xb") as handle:
    handle.write(raw)
keep = ["CUDA_HOME", "CUDA_PATH", "LD_LIBRARY_PATH", "LIBRARY_PATH", "NVIDIA_DRIVER_CAPABILITIES", "NVIDIA_VISIBLE_DEVICES", "PATH"]
environment = {key: os.environ[key] for key in keep if key in os.environ}
environment.update({
    "HOME": str(WORK),
    "TMPDIR": str(WORK),
    "PIP_CONFIG_FILE": os.devnull,
    "PIP_NO_INDEX": "1",
    "PIP_DISABLE_PIP_VERSION_CHECK": "1",
    "PYTHONNOUSERSITE": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "PYTHONSAFEPATH": "1",
})
python = Path(sys.executable).resolve(strict=True)
prefixes = {Path(sys.prefix).resolve(strict=True), Path(sys.base_prefix).resolve(strict=True)}
if not python.is_file() or not any(python == root or root in python.parents for root in prefixes):
    raise RuntimeError("Python executable escaped its runtime prefix")
result = subprocess.run(
    [str(python), "-P", str(CODE)], check=False, cwd=str(WORK), env=environment
)
if result.returncode != 0:
    raise RuntimeError("bh9gb failed closed; inspect /kaggle/working/bh9fail.json")
print("bh9gb completed; evidence=/kaggle/working/bh9.json run=/kaggle/working/bh9out/run.json fragments=/kaggle/working/bh9out/fragments")
